In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.kayashev.lesson3 import Exercise

In [ ]:
digits = load_digits()
print(digits.data.shape)
x = digits.data
y = digits.target
plt.matshow(digits.images[16], cmap="gray")
plt.show()
xtrain, xvalidtest, ytrain, yvalidtest = train_test_split(x, y, test_size=0.4)
xvalid, xtest, yvalid, ytest = train_test_split(xvalidtest, yvalidtest, test_size=0.5)
means = xtrain.mean(axis=0)
stds = xtrain.std(axis=0)
non_zero_std = stds > 1e-8
xtrain[:, non_zero_std] = (xtrain[:, non_zero_std] - means[non_zero_std]) / stds[non_zero_std]
xvalid[:, non_zero_std] = (xvalid[:, non_zero_std] - means[non_zero_std]) / stds[non_zero_std]
xtest[:, non_zero_std] = (xtest[:, non_zero_std] - means[non_zero_std]) / stds[non_zero_std]

In [ ]:
lrs = 1 / 2 ** np.arange(7)
epochs = 25
batches = 2 ** np.arange(7)
hiddenlairdim = 32
print(lrs)
print(len(np.unique(ytrain)))

In [ ]:
def testparams(lr, batches, xtrain, xvalid, ytrain, yvalid, epochs):
    rng = np.random.default_rng()
    model = Exercise.create_model(
        Exercise.create_linear_layer(xtrain.shape[1], hiddenlairdim, rng),
        Exercise.create_sigmoid_layer(),
        Exercise.create_linear_layer(hiddenlairdim, len(np.unique(ytrain)), rng),
    )
    loss = Exercise.create_cross_entropy_loss()
    Exercise.train_model(
        model=model,
        loss=loss,
        x=xtrain,
        y=ytrain,
        lr=lr,
        n_epoch=epochs,
        batch_size=batches,
        shuffle=True,
    )
    predict = model.forward(xvalid)
    # if lr == 1 and (batches == 1 or batches == 2):
    #    print(model.parameters[0][0], "\n")
    #    print("Predict: ", predict, "\n")
    predicty = np.argmax(predict, axis=1)
    accuracy = np.mean(predicty == yvalid)

    return accuracy, model

In [ ]:
hyperparamacc = []
for lr in lrs:
    for batch in batches:
        acc, model = testparams(lr, batch, xtrain, xvalid, ytrain, yvalid, epochs)
        print(lr, " ", batch, " ", acc, "\n")
        hyperparamacc.append(tuple((acc, model, batch, lr)))

In [ ]:
hyperparamacc.sort(key=lambda x: x[0])
bestmodel = hyperparamacc[-1]
print("Best model hyperparams: \n")
print("Batches: ", bestmodel[2], "\n")
print("Learning rate: ", bestmodel[3], "\n")
print("Accuracy: ", bestmodel[0], "\n")